# Face2Comic — pix2pix Hyperparameter Tuning & Training

This notebook is a **self-contained** pipeline for training a pix2pix conditional GAN that converts real face photos into comic-style images.

**What's inside:**
1. Data loading (pre-processed `.npy` arrays, already normalized to [-1, 1])
2. U-Net Generator + PatchGAN Discriminator (built from scratch)
3. Hyperparameter grid search over learning rate, batch size, and L1 weight
4. Visualization of grid search results (charts, heatmaps, sample grids)
5. Full training run with the best configuration
6. Training curves, sample progression, and final evaluation

In [ ]:
import os
import csv
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchinfo import summary

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

NUM_WORKERS = 0

print(f"Random seed: {SEED}")
print(f"Number of workers: {NUM_WORKERS}")

## Device Selection

Use CUDA if available, otherwise fall back to MPS (Apple Silicon) or CPU.

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")

## 1. Load Data

The preprocessing notebook already resized all images to 256x256, converted them to CHW format, and normalized pixel values to [-1, 1]. We load them as memory-mapped arrays so only the batches we need are read into RAM.

In [ ]:
train_real = np.load("../data/npy/train_real.npy", mmap_mode="r")
train_comic = np.load("../data/npy/train_comic.npy", mmap_mode="r")

val_real = np.load("../data/npy/val_real.npy", mmap_mode="r")
val_comic = np.load("../data/npy/val_comic.npy", mmap_mode="r")

test_real = np.load("../data/npy/test_real.npy", mmap_mode="r")
test_comic = np.load("../data/npy/test_comic.npy", mmap_mode="r")

print(f"Train — real: {train_real.shape}, comic: {train_comic.shape}")
print(f"Val   — real: {val_real.shape},   comic: {val_comic.shape}")
print(f"Test  — real: {test_real.shape},  comic: {test_comic.shape}")
print(f"Pixel range: [{train_real[0].min():.1f}, {train_real[0].max():.1f}]")

## 2. Dataset & DataLoader

A thin wrapper that converts memory-mapped NumPy rows into PyTorch tensors on the fly. For the grid search we use a **2 000-sample training subset** and a **500-sample validation subset** to keep each trial fast (~2 min). The full training and validation splits are used for final model training, and the held-out test split is reserved for final evaluation only.

In [ ]:
class Face2ComicDataset(Dataset):
    def __init__(self, real, comic):
        self.real = real
        self.comic = comic

    def __len__(self):
        return len(self.real)

    def __getitem__(self, idx):
        real_img = self.real[idx].astype(np.float32, copy=True)
        comic_img = self.comic[idx].astype(np.float32, copy=True)
        return torch.from_numpy(real_img), torch.from_numpy(comic_img)


train_dataset = Face2ComicDataset(train_real, train_comic)
full_val_dataset = Face2ComicDataset(val_real, val_comic)
full_test_dataset = Face2ComicDataset(test_real, test_comic)

# Smaller subsets so the grid search finishes in reasonable time
TRAIN_TUNING_SAMPLES = 2000
VAL_TUNING_SAMPLES = 500
train_tuning_dataset = Subset(train_dataset, range(min(TRAIN_TUNING_SAMPLES, len(train_dataset))))
val_tuning_dataset = Subset(full_val_dataset, range(min(VAL_TUNING_SAMPLES, len(full_val_dataset))))

print(f"Train samples (full): {len(train_dataset)}")
print(f"Train samples (tuning subset): {len(train_tuning_dataset)}")
print(f"Val samples (full): {len(full_val_dataset)}")
print(f"Val samples (tuning subset): {len(val_tuning_dataset)}")
print(f"Test samples (full): {len(full_test_dataset)}")

### Helper to build DataLoaders for any batch size

In [ ]:
def make_loaders(batch_size, train_ds=train_tuning_dataset, val_ds=val_tuning_dataset):
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    return train_loader, val_loader

## 3. Generator — U-Net

The generator is an encoder-decoder with skip connections. The encoder compresses the 256x256 input down to a 4x4 bottleneck, and the decoder upsamples it back. Skip connections carry high-frequency spatial detail (edges, facial features) from encoder to decoder so they survive the bottleneck.

- **Encoder blocks**: Conv2d(stride=2) → InstanceNorm → LeakyReLU
- **Decoder blocks**: ConvTranspose2d(stride=2) → InstanceNorm → ReLU (+ Dropout in first two)
- **Output**: Tanh activation → pixel values in [-1, 1]

In [ ]:
class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)

In [ ]:
class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3):
        super().__init__()
        # Encoder: 256→128→64→32→16→8→4
        self.d1 = UNetDown(in_ch, 64, normalize=False)  # 256→128
        self.d2 = UNetDown(64, 128)                      # 128→64
        self.d3 = UNetDown(128, 256)                     # 64→32
        self.d4 = UNetDown(256, 512)                     # 32→16
        self.d5 = UNetDown(512, 512)                     # 16→8
        self.d6 = UNetDown(512, 512)                     # 8→4  (bottleneck)

        # Decoder: 4→8→16→32→64→128→256
        self.u1 = UNetUp(512, 512, dropout=True)         # 4→8,   cat d5 → 1024
        self.u2 = UNetUp(1024, 512, dropout=True)        # 8→16,  cat d4 → 1024
        self.u3 = UNetUp(1024, 256)                      # 16→32, cat d3 → 512
        self.u4 = UNetUp(512, 128)                       # 32→64, cat d2 → 256
        self.u5 = UNetUp(256, 64)                        # 64→128, cat d1 → 128

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_ch, 4, 2, 1),    # 128→256
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)

        u1 = self.u1(d6, d5)
        u2 = self.u2(u1, d4)
        u3 = self.u3(u2, d3)
        u4 = self.u4(u3, d2)
        u5 = self.u5(u4, d1)

        return self.final(u5)

In [ ]:
generator = Generator()
summary(generator, input_size=(1, 3, 256, 256))

## 4. Discriminator — PatchGAN

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_ch=6):
        super().__init__()
        def block(in_c, out_c, normalize=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_ch, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, real_input, target_or_fake):
        x = torch.cat([real_input, target_or_fake], dim=1)
        return self.model(x)

In [ ]:
discriminator = Discriminator()
summary(discriminator, input_size=[(1, 3, 256, 256), (1, 3, 256, 256)])

### Weight Initialization

Standard pix2pix initialization: draw conv weights from N(0, 0.02) and set BatchNorm weights to N(1, 0.02) with zero bias.

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


def make_models(lr, betas=(0.5, 0.999)):
    gen = Generator().to(DEVICE)
    disc = Discriminator().to(DEVICE)
    gen.apply(init_weights)
    disc.apply(init_weights)
    opt_gen = optim.Adam(gen.parameters(), lr=lr, betas=betas)
    opt_disc = optim.Adam(disc.parameters(), lr=lr, betas=betas)
    return gen, disc, opt_gen, opt_disc


_g = Generator()
_d = Discriminator()
print(f"Generator params:     {sum(p.numel() for p in _g.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in _d.parameters()):,}")
del _g, _d

## 5. Loss Functions

Two losses work together to make generated comics as close as possible to the ground truth:

| Loss | Purpose |
|------|---------|
| **BCE (adversarial)** | Fools the discriminator — makes outputs look "real" |
| **L1 (pixel)** | Penalizes pixel-level deviation from the target comic |

The generator loss is: `G_loss = BCE + λ_L1 × L1`

In [ ]:
bce = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()

## 6. Training & Validation Functions

**Training step (per batch):**
1. Generate a fake comic from the real face
2. Train the **Discriminator** — show it (real, target) as "real" and (real, fake) as "fake"
3. Train the **Generator** — fool the discriminator + minimize L1 + minimize LPIPS

**Validation:** just compute average L1 loss on held-out pairs (no gradient updates).

In [ ]:
def train_one_epoch(gen, disc, loader, opt_gen, opt_disc, lambda_l1):
    gen.train()
    disc.train()

    total_g, total_d = 0.0, 0.0
    loop = tqdm(loader, leave=True)

    for real, target in loop:
        real = real.to(DEVICE)
        target = target.to(DEVICE)

        fake = gen(real)

        # ── Train Discriminator ──
        pred_real = disc(real, target)
        loss_real = bce(pred_real, torch.ones_like(pred_real))

        pred_fake = disc(real, fake.detach())
        loss_fake = bce(pred_fake, torch.zeros_like(pred_fake))

        loss_d = (loss_real + loss_fake) / 2

        opt_disc.zero_grad()
        loss_d.backward()
        opt_disc.step()

        # ── Train Generator ──
        pred_fake = disc(real, fake)
        loss_g_gan = bce(pred_fake, torch.ones_like(pred_fake))
        loss_g_l1 = l1_loss(fake, target)

        loss_g = loss_g_gan + lambda_l1 * loss_g_l1

        opt_gen.zero_grad()
        loss_g.backward()
        opt_gen.step()

        total_g += loss_g.item()
        total_d += loss_d.item()

        loop.set_postfix(G=f"{loss_g.item():.3f}", D=f"{loss_d.item():.3f}")

    return total_g / len(loader), total_d / len(loader)

In [ ]:
def validate(gen, loader):
    gen.eval()
    total_l1 = 0.0

    with torch.no_grad():
        for real, target in loader:
            real = real.to(DEVICE)
            target = target.to(DEVICE)
            fake = gen(real)
            total_l1 += l1_loss(fake, target).item()

    return total_l1 / len(loader)

## 7. Hyperparameter Grid Search

We search over **3 learning rates × 3 batch sizes × 3 L1 weights = 27 configurations**. Each one trains for 10 short epochs on a **2 000-sample training subset** and is validated on the 500-sample tuning subset — this keeps each trial fast (~2 min).

The CSV log is written after every config so nothing is lost if the kernel dies. The best config (lowest validation L1) gets its checkpoint saved.

In [ ]:
GRID_LRS = [1e-4, 2e-4, 5e-4]
GRID_BATCH_SIZES = [16, 32, 64]
GRID_LAMBDA_L1 = [50, 100, 150]
GRID_EPOCHS = 10
GRID_BETAS = (0.5, 0.999)
GRID_PREVIEW_INDICES = [9, 10, 11]

LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

GRID_SAMPLE_DIR = Path("../outputs/grid_search_samples")
GRID_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

total_configs = len(GRID_LRS) * len(GRID_BATCH_SIZES) * len(GRID_LAMBDA_L1)
print(f"Grid search: {total_configs} configurations, {GRID_EPOCHS} epochs each")

In [ ]:
def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)


def save_grid_samples(gen, dataset, config_idx, sample_indices=None):
    gen.eval()
    sample_indices = GRID_PREVIEW_INDICES if sample_indices is None else sample_indices
    valid_indices = [idx for idx in sample_indices if 0 <= idx < len(dataset)]
    if not valid_indices:
        valid_indices = list(range(min(3, len(dataset))))

    fig, axes = plt.subplots(len(valid_indices), 3, figsize=(9, 3 * len(valid_indices)), squeeze=False)
    axes[0, 0].set_title("Real Face")
    axes[0, 1].set_title("Generated")
    axes[0, 2].set_title("Ground Truth")

    with torch.no_grad():
        for row, idx in enumerate(valid_indices):
            real, target = dataset[idx]
            real_gpu = real.unsqueeze(0).to(DEVICE)
            fake = gen(real_gpu).squeeze(0).cpu()

            axes[row, 0].imshow(denorm(real).permute(1, 2, 0).numpy())
            axes[row, 1].imshow(denorm(fake).permute(1, 2, 0).numpy())
            axes[row, 2].imshow(denorm(target).permute(1, 2, 0).numpy())

            for j in range(3):
                axes[row, j].axis("off")

    plt.tight_layout()
    path = GRID_SAMPLE_DIR / f"config_{config_idx:02d}.png"
    fig.savefig(path, dpi=100)
    plt.close(fig)
    return path

In [ ]:
GRID_CSV = LOG_DIR / "grid_search_results.csv"

with open(GRID_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["config_idx", "learning_rate", "batch_size", "lambda_l1", "val_l1"])
    writer.writeheader()

grid_results = []
best_val_l1 = float("inf")
best_grid_state = None
config_idx = 0

for lr in GRID_LRS:
    for bs in GRID_BATCH_SIZES:
        for lam in GRID_LAMBDA_L1:
            config_idx += 1
            print(f"\n{'='*60}")
            print(f"Config {config_idx}/{total_configs}  LR={lr:.0e}  BS={bs}  λ_L1={lam}")
            print(f"{'='*60}")

            gen, disc, opt_gen, opt_disc = make_models(lr, GRID_BETAS)
            train_loader, val_loader = make_loaders(bs)

            for epoch in range(1, GRID_EPOCHS + 1):
                g_loss, d_loss = train_one_epoch(gen, disc, train_loader, opt_gen, opt_disc, lam)
                if epoch % 5 == 0 or epoch == GRID_EPOCHS:
                    print(f"  Epoch {epoch}/{GRID_EPOCHS}  G={g_loss:.4f}  D={d_loss:.4f}")

            val_l1 = validate(gen, val_loader)
            print(f"  Val L1: {val_l1:.6f}")

            save_grid_samples(gen, full_val_dataset, config_idx)
            
            # show a quick 3-sample preview inline
            preview_indices = [idx for idx in GRID_PREVIEW_INDICES if 0 <= idx < len(full_val_dataset)]
            if not preview_indices:
                preview_indices = list(range(min(3, len(full_val_dataset))))

            gen.eval()
            fig, axes = plt.subplots(len(preview_indices), 3, figsize=(9, 3 * len(preview_indices)), squeeze=False)
            axes[0, 0].set_title("Real")
            axes[0, 1].set_title("Generated")
            axes[0, 2].set_title("Target")
            with torch.no_grad():
                for row, idx in enumerate(preview_indices):
                    real_s, target_s = full_val_dataset[idx]
                    fake_s = gen(real_s.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
                    axes[row, 0].imshow(denorm(real_s).permute(1, 2, 0).numpy())
                    axes[row, 1].imshow(denorm(fake_s).permute(1, 2, 0).numpy())
                    axes[row, 2].imshow(denorm(target_s).permute(1, 2, 0).numpy())
                    for j in range(3):
                        axes[row, j].axis("off")
            fig.suptitle(f"Config {config_idx}  LR={lr:.0e}  BS={bs}  λ={lam}  Val L1={val_l1:.4f}", fontweight="bold")
            plt.tight_layout()
            plt.show()

            row = {"config_idx": config_idx, "learning_rate": lr, "batch_size": bs, "lambda_l1": lam, "val_l1": round(val_l1, 6)}
            grid_results.append(row)

            with open(GRID_CSV, "a", newline="") as f:
                csv.DictWriter(f, fieldnames=row.keys()).writerow(row)

            if val_l1 < best_val_l1:
                best_val_l1 = val_l1
                best_grid_state = {
                    "config_idx": config_idx,
                    "lr": lr, "bs": bs, "lambda_l1": lam,
                    "val_l1": val_l1,
                    "gen_state_dict": gen.state_dict(),
                    "disc_state_dict": disc.state_dict(),
                }

            del gen, disc, opt_gen, opt_disc
            torch.cuda.empty_cache()

# save best grid checkpoint
torch.save(best_grid_state, CHECKPOINT_DIR / "best_grid_config.pt")

print(f"\n{'='*60}")
print("Grid Search Complete!")
print(f"{'='*60}")
print(f"\nBest Config #{best_grid_state['config_idx']}  (Val L1 = {best_val_l1:.6f})")
print(f"  LR:       {best_grid_state['lr']:.0e}")
print(f"  Batch:    {best_grid_state['bs']}")
print(f"  λ_L1:     {best_grid_state['lambda_l1']}")
print(f"\nResults saved to {GRID_CSV}")

## 8. Grid Search — Visual Analysis

Three views of the results:
1. **Bar chart** — Val L1 for every configuration, color-coded by learning rate
2. **Heatmap** — Val L1 as a function of (LR × λ_L1), averaged over batch sizes
3. **Sample grid** — Real / Generated / Ground Truth from the top-3 configurations

In [ ]:
# ── Bar chart: Val L1 per config, colored by LR ──
sorted_results = sorted(grid_results, key=lambda r: r["config_idx"])
indices = [r["config_idx"] for r in sorted_results]
val_l1s = [r["val_l1"] for r in sorted_results]
lrs = [r["learning_rate"] for r in sorted_results]

lr_colors = {1e-4: "#1f77b4", 2e-4: "#ff7f0e", 5e-4: "#2ca02c"}
colors = [lr_colors[lr] for lr in lrs]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(indices, val_l1s, color=colors)
ax.set_xlabel("Configuration #")
ax.set_ylabel("Validation L1 Loss")
ax.set_title("Grid Search Results — Validation L1 by Configuration")

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f"LR={k:.0e}") for k, c in lr_colors.items()]
ax.legend(handles=legend_elements)

best_idx = min(sorted_results, key=lambda r: r["val_l1"])["config_idx"]
ax.axvline(best_idx, color="red", linestyle="--", alpha=0.7, label="Best")
ax.annotate("Best", xy=(best_idx, min(val_l1s)), fontsize=10, color="red",
            ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: Val L1 averaged over batch sizes, as (LR × λ_L1) ──
heatmap_data = {}
for r in grid_results:
    key = (r["learning_rate"], r["lambda_l1"])
    heatmap_data.setdefault(key, []).append(r["val_l1"])

lr_vals = sorted(set(r["learning_rate"] for r in grid_results))
lam_vals = sorted(set(r["lambda_l1"] for r in grid_results))

heat = np.zeros((len(lr_vals), len(lam_vals)))
for i, lr in enumerate(lr_vals):
    for j, lam in enumerate(lam_vals):
        heat[i, j] = np.mean(heatmap_data[(lr, lam)])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heat, cmap="YlOrRd_r", aspect="auto")
ax.set_xticks(range(len(lam_vals)))
ax.set_xticklabels([str(int(v)) for v in lam_vals])
ax.set_yticks(range(len(lr_vals)))
ax.set_yticklabels([f"{v:.0e}" for v in lr_vals])
ax.set_xlabel("λ_L1")
ax.set_ylabel("Learning Rate")
ax.set_title("Average Val L1 (lower = better)")

for i in range(len(lr_vals)):
    for j in range(len(lam_vals)):
        ax.text(j, i, f"{heat[i, j]:.4f}", ha="center", va="center", fontsize=10)

fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sample grid from top-3 configs ──
top3 = sorted(grid_results, key=lambda r: r["val_l1"])[:3]

for rank, r in enumerate(top3, 1):
    img_path = GRID_SAMPLE_DIR / f"config_{r['config_idx']:02d}.png"
    if img_path.exists():
        img = Image.open(img_path)
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.imshow(img)
        ax.set_title(f"Rank #{rank} — Config {r['config_idx']}  "
                     f"(LR={r['learning_rate']:.0e}, BS={r['batch_size']}, λ={r['lambda_l1']}, "
                     f"Val L1={r['val_l1']:.4f})")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

## 9. Full Training with Best Configuration

Now we train for **200 epochs** using the winning hyperparameters from the grid search. Key additions:

- **LR scheduling**: linear decay from the original LR to 0 over the second half of training (epochs 101–200) for stable convergence
- **Checkpoints**: saved every 10 epochs + the single best model (by validation L1)
- **CSV logging**: every epoch is appended immediately

### Checkpoint save / load helpers

In [ ]:
def save_checkpoint(gen, disc, opt_gen, opt_disc, epoch, metrics, name="pix2pix"):
    ckpt = {
        "epoch": epoch,
        "gen_state_dict": gen.state_dict(),
        "disc_state_dict": disc.state_dict(),
        "opt_gen_state_dict": opt_gen.state_dict(),
        "opt_disc_state_dict": opt_disc.state_dict(),
        "metrics": metrics,
    }
    path = CHECKPOINT_DIR / f"{name}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, path)
    return path


def load_checkpoint(path, gen, disc, opt_gen=None, opt_disc=None):
    ckpt = torch.load(path, map_location=DEVICE)
    gen.load_state_dict(ckpt["gen_state_dict"])
    disc.load_state_dict(ckpt["disc_state_dict"])
    if opt_gen is not None:
        opt_gen.load_state_dict(ckpt["opt_gen_state_dict"])
    if opt_disc is not None:
        opt_disc.load_state_dict(ckpt["opt_disc_state_dict"])
    return ckpt["epoch"], ckpt.get("metrics", {})

### Set up from best grid config

In [ ]:
best = min(grid_results, key=lambda r: r["val_l1"])
BEST_LR = best["learning_rate"]
BEST_BS = best["batch_size"]
BEST_LAMBDA_L1 = best["lambda_l1"]
NUM_EPOCHS = 200
CHECKPOINT_EVERY = 10
DECAY_START = 100

print("Full training configuration:")
print(f"  LR:       {BEST_LR:.0e}")
print(f"  Batch:    {BEST_BS}")
print(f"  λ_L1:     {BEST_LAMBDA_L1}")
print(f"  Epochs:   {NUM_EPOCHS}")
print(f"  LR decay: linear from epoch {DECAY_START}")

gen, disc, opt_gen, opt_disc = make_models(BEST_LR)
train_loader, val_loader = make_loaders(BEST_BS, train_ds=train_dataset, val_ds=full_val_dataset)

# LR schedulers: hold constant for DECAY_START epochs, then linearly decay to 0
def lr_lambda(epoch):
    if epoch < DECAY_START:
        return 1.0
    return 1.0 - (epoch - DECAY_START) / (NUM_EPOCHS - DECAY_START)

sched_gen = optim.lr_scheduler.LambdaLR(opt_gen, lr_lambda)
sched_disc = optim.lr_scheduler.LambdaLR(opt_disc, lr_lambda)

### Training loop

In [ ]:
TRAIN_CSV = LOG_DIR / "full_training_log.csv"

with open(TRAIN_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "g_loss", "d_loss", "val_l1", "lr", "time_sec"])
    writer.writeheader()

history = {"epoch": [], "g_loss": [], "d_loss": [], "val_l1": [], "lr": []}
best_val = float("inf")
best_epoch = 0

# directory to save sample snapshots during training
EPOCH_SAMPLE_DIR = Path("../outputs/epoch_samples")
EPOCH_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

snapshot_epochs = {1, 25, 50, 75, 100}

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    g_loss, d_loss = train_one_epoch(gen, disc, train_loader, opt_gen, opt_disc, BEST_LAMBDA_L1)
    val_l1 = validate(gen, val_loader)

    sched_gen.step()
    sched_disc.step()
    current_lr = opt_gen.param_groups[0]["lr"]
    elapsed = time.time() - t0

    history["epoch"].append(epoch)
    history["g_loss"].append(g_loss)
    history["d_loss"].append(d_loss)
    history["val_l1"].append(val_l1)
    history["lr"].append(current_lr)

    row = {"epoch": epoch, "g_loss": round(g_loss, 6), "d_loss": round(d_loss, 6),
           "val_l1": round(val_l1, 6), "lr": round(current_lr, 8), "time_sec": round(elapsed, 2)}
    with open(TRAIN_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=row.keys()).writerow(row)

    print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  G={g_loss:.4f}  D={d_loss:.4f}  "
          f"Val_L1={val_l1:.4f}  LR={current_lr:.6f}  ({elapsed:.1f}s)")

    # save periodic checkpoint
    if epoch % CHECKPOINT_EVERY == 0:
        p = save_checkpoint(gen, disc, opt_gen, opt_disc, epoch,
                            {"g_loss": g_loss, "d_loss": d_loss, "val_l1": val_l1})
        print(f"  ✓ Checkpoint saved: {p.name}")

    # save best model
    if val_l1 < best_val:
        best_val = val_l1
        best_epoch = epoch
        save_checkpoint(gen, disc, opt_gen, opt_disc, epoch,
                        {"g_loss": g_loss, "d_loss": d_loss, "val_l1": val_l1},
                        name="pix2pix_best")
        print(f"  ★ New best model (Val L1={val_l1:.4f})")

    # save sample snapshots at milestone epochs
    if epoch in snapshot_epochs:
        gen.eval()
        with torch.no_grad():
            fig, axes = plt.subplots(4, 3, figsize=(9, 12))
            axes[0, 0].set_title("Real")
            axes[0, 1].set_title("Generated")
            axes[0, 2].set_title("Target")
            for i in range(4):
                real, target = full_val_dataset[i]
                fake = gen(real.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()
                axes[i, 0].imshow(denorm(real).permute(1, 2, 0).numpy())
                axes[i, 1].imshow(denorm(fake).permute(1, 2, 0).numpy())
                axes[i, 2].imshow(denorm(target).permute(1, 2, 0).numpy())
                for j in range(3):
                    axes[i, j].axis("off")
            fig.suptitle(f"Epoch {epoch}", fontsize=14, fontweight="bold")
            plt.tight_layout()
            fig.savefig(EPOCH_SAMPLE_DIR / f"epoch_{epoch:03d}.png", dpi=100)
            plt.close(fig)

print(f"\nTraining complete! Best Val L1 = {best_val:.4f} at epoch {best_epoch}")

## 10. Training Visualization Dashboard

Four plots to understand how training went:
1. **G and D loss curves** (dual y-axis) — are they balanced?
2. **Validation L1 curve** — is the generator improving?
3. **Learning rate schedule** — confirms the linear decay kicked in
4. **Sample progression** — visual evolution at epochs 1, 25, 50, 75, 100

In [ ]:
epochs = history["epoch"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: G loss and D loss ──
ax1 = axes[0]
ax1.plot(epochs, history["g_loss"], label="G Loss", color="tab:blue")
ax1_twin = ax1.twinx()
ax1_twin.plot(epochs, history["d_loss"], label="D Loss", color="tab:orange")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Generator Loss", color="tab:blue")
ax1_twin.set_ylabel("Discriminator Loss", color="tab:orange")
ax1.set_title("Generator vs Discriminator Loss")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

# ── Plot 2: Validation L1 ──
ax2 = axes[1]
ax2.plot(epochs, history["val_l1"], color="tab:green")
ax2.axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
ax2.annotate(f"Best: {best_val:.4f}\n(epoch {best_epoch})",
             xy=(best_epoch, best_val), fontsize=9, color="red",
             ha="left", va="bottom")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation L1")
ax2.set_title("Validation L1 Loss")

# ── Plot 3: Learning rate ──
ax3 = axes[2]
ax3.plot(epochs, history["lr"], color="tab:purple")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Learning Rate")
ax3.set_title("LR Schedule (linear decay)")

plt.tight_layout()
plt.show()

### Sample Progression

Showing how the generator's output evolves over training.

In [ ]:
snapshot_files = sorted(EPOCH_SAMPLE_DIR.glob("epoch_*.png"))

if snapshot_files:
    fig, axes = plt.subplots(1, len(snapshot_files), figsize=(5 * len(snapshot_files), 12))
    if len(snapshot_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, snapshot_files):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(path.stem.replace("_", " ").title(), fontsize=12)
        ax.axis("off")
    plt.suptitle("Generator Output Over Training", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No snapshot images found.")

## 11. Final Evaluation

Load the best checkpoint and generate comic images on 8 unseen **test** samples. Each row shows: **Real Face → Generated Comic → Ground Truth Comic**.

In [ ]:
# load best model
best_ckpt_path = CHECKPOINT_DIR / f"pix2pix_best_epoch_{best_epoch:03d}.pt"
eval_gen = Generator().to(DEVICE)
eval_disc = Discriminator().to(DEVICE)
loaded_epoch, loaded_metrics = load_checkpoint(best_ckpt_path, eval_gen, eval_disc)
print(f"Loaded best model from epoch {loaded_epoch}  (Best Val L1 = {loaded_metrics.get('val_l1', 'N/A')})")

eval_gen.eval()
NUM_EVAL_SAMPLES = 8
n_eval = min(NUM_EVAL_SAMPLES, len(full_test_dataset))

fig, axes = plt.subplots(n_eval, 3, figsize=(12, 4 * n_eval), squeeze=False)
axes[0, 0].set_title("Real Face", fontsize=13, fontweight="bold")
axes[0, 1].set_title("Generated Comic", fontsize=13, fontweight="bold")
axes[0, 2].set_title("Ground Truth", fontsize=13, fontweight="bold")

with torch.no_grad():
    for i in range(n_eval):
        # take samples from the end of the held-out test split
        idx = len(full_test_dataset) - 1 - i
        real, target = full_test_dataset[idx]
        fake = eval_gen(real.unsqueeze(0).to(DEVICE)).squeeze(0).cpu()

        axes[i, 0].imshow(denorm(real).permute(1, 2, 0).numpy())
        axes[i, 1].imshow(denorm(fake).permute(1, 2, 0).numpy())
        axes[i, 2].imshow(denorm(target).permute(1, 2, 0).numpy())

        for j in range(3):
            axes[i, j].axis("off")

plt.suptitle(f"Best Model on Held-Out Test Samples — Epoch {loaded_epoch}", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### Compute average L1 on full test set

In [ ]:
eval_loader = DataLoader(full_test_dataset, batch_size=16, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

eval_gen.eval()
total_l1, n_batches = 0.0, 0

with torch.no_grad():
    for real, target in tqdm(eval_loader, desc="Evaluating on test set"):
        real = real.to(DEVICE)
        target = target.to(DEVICE)
        fake = eval_gen(real)
        total_l1 += l1_loss(fake, target).item()
        n_batches += 1

avg_l1 = total_l1 / n_batches

print(f"\nFinal Evaluation (held-out test set):")
print(f"  Average L1 Loss: {avg_l1:.4f}")

## 12. Save Training Summary

Export a JSON file capturing everything: grid search config, best hyperparameters, final metrics, and paths to all saved artifacts. This makes it easy to pick up where we left off or report results.

In [ ]:
summary = {
    "hyperparameter_tuning": {
        "total_configs": total_configs,
        "grid_epochs": GRID_EPOCHS,
        "grid_search_file": str(GRID_CSV),
    },
    "best_configuration": {
        "learning_rate": BEST_LR,
        "batch_size": BEST_BS,
        "lambda_l1": BEST_LAMBDA_L1,
    },
    "full_training": {
        "epochs": NUM_EPOCHS,
        "lr_decay_start": DECAY_START,
        "best_epoch": best_epoch,
        "best_val_l1": round(best_val, 6),
    },
    "final_evaluation": {
        "split": "test",
        "test_samples": len(full_test_dataset),
        "avg_l1": round(avg_l1, 6),
    },
    "artifacts": {
        "best_model": str(best_ckpt_path),
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "training_log": str(TRAIN_CSV),
        "grid_search_log": str(GRID_CSV),
        "epoch_samples": str(EPOCH_SAMPLE_DIR),
    },
}

summary_path = LOG_DIR / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to {summary_path}")
print(json.dumps(summary, indent=2))